In [ ]:
import subprocess
import sys

JOB_PACKAGES = [
    "huggingface_hub",
    "pyarrow",
    "pandas",
    "ray[data]",
    "requests",
    "tqdm",
    "torch",
    "transformers",
    "accelerate",
    "tokenizers",
    "sentencepiece",
    "protobuf",
    "safetensors",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *JOB_PACKAGES])
print("Dependencies installed in notebook kernel. Re-run this cell only if needed.")

In [ ]:
RUNTIME_PIP_PACKAGES = [
    "huggingface_hub",
    "pyarrow",
    "pandas",
    "ray[data]",
    "requests",
    "tqdm",
    "torch",
    "transformers",
    "accelerate",
    "tokenizers",
    "sentencepiece",
    "protobuf",
    "safetensors",
]

print("Runtime dependency list prepared. Step 1 will use it during ray.init(...).")

# Ray Distributed Training Pipeline

This notebook executes the scale_train job step-by-step:
1. **Configuration & Imports** — Set up constants and connect to Ray cluster
2. **Storage Validation** — Verify shared storage is mounted
3. **Model Download** — Download Mistral-7B to shared storage
4. **Dataset Download** — Distribute Fineweb-Edu download across workers
5. **Validate Downloads** — Verify model and dataset integrity
6. **Distributed Training** — 15 workers × 10 epochs with per-epoch checkpoints
7. **Model Consolidation** — Aggregate checkpoints into final model
8. **Job Summary & Cleanup** — Collect logs and report results

## Step 0: Imports & Configuration

In [ ]:
import os
import time
import sys
import json
import socket
import ray
from huggingface_hub import snapshot_download, hf_hub_download, list_repo_files

# ------------------------------------------------------------------------------
# Configuration & Constants
# ------------------------------------------------------------------------------
STORAGE_PATH = os.environ.get("STORAGE_PATH", "/mnt/shared_storage")
MODEL_SAVE_PATH = os.path.join(STORAGE_PATH, "model")
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
DATASET_ID = "HuggingFaceFW/fineweb-edu"
DATASET_SUBSET = "sample/100BT"
DATASET_SAVE_PATH = os.path.join(STORAGE_PATH, "dataset")
CHECKPOINT_SAVE_PATH = os.path.join(STORAGE_PATH, "checkpoint")

# Logging setup
_JOB_TIMESTAMP = int(time.time())
_HOSTNAME = socket.gethostname()
LOG_DIR = "/tmp/ray_job_logs"
os.makedirs(LOG_DIR, exist_ok=True)
LOG_FILE = os.path.join(LOG_DIR, f"job_{_JOB_TIMESTAMP}_{_HOSTNAME}.log")

def log(message):
    """Prints to stdout AND appends to a local log file."""
    print(message, flush=True)
    try:
        with open(LOG_FILE, "a") as f:
            f.write(message + "\n")
            f.flush()
    except Exception:
        pass

print(f"Configuration:")
print(f"  STORAGE_PATH:         {STORAGE_PATH}")
print(f"  MODEL_SAVE_PATH:      {MODEL_SAVE_PATH}")
print(f"  MODEL_ID:             {MODEL_ID}")
print(f"  DATASET_ID:           {DATASET_ID}")
print(f"  DATASET_SAVE_PATH:    {DATASET_SAVE_PATH}")
print(f"  CHECKPOINT_SAVE_PATH: {CHECKPOINT_SAVE_PATH}")
print(f"  LOG_FILE:             {LOG_FILE}")

## Step 1: Initialize Ray Cluster

In [ ]:
log("\n[Init] Initializing Ray Cluster Connection...")

runtime_packages = globals().get("RUNTIME_PIP_PACKAGES", None)

if ray.is_initialized():
    log("[Init] Ray is already initialized.")
else:
    if runtime_packages:
        log(f"[Init] Applying runtime_env pip dependencies ({len(runtime_packages)} packages)...")
        ray.init(address="auto", log_to_driver=True, runtime_env={"pip": runtime_packages})
    else:
        ray.init(address="auto", log_to_driver=True)

resources = ray.available_resources()
log(f"[Init] Connected. Cluster Resources: {resources}")

## Step 2: Validate & Setup Storage

In [ ]:
def validate_and_setup_storage():
    """Validates shared storage path exists and creates subdirectories."""
    log(f"[Init] Checking storage path: {STORAGE_PATH}")

    if not os.path.exists(STORAGE_PATH):
        error_msg = f"[CRITICAL] Storage path {STORAGE_PATH} not found! Ensure PV is mounted."
        log(error_msg)
        raise FileNotFoundError(error_msg)

    log(f"[Init] Storage path confirmed: {STORAGE_PATH}")

    try:
        log(f"[Init] Creating directories in {STORAGE_PATH}...")
        os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
        os.makedirs(DATASET_SAVE_PATH, exist_ok=True)
        os.makedirs(CHECKPOINT_SAVE_PATH, exist_ok=True)
        log(f"[Init] Directories created successfully.")
    except Exception as e:
        log(f"[CRITICAL] Failed to create directories: {e}")
        raise

validate_and_setup_storage()

## Step 3: Download Model

Downloads Mistral-7B-Instruct-v0.2 (~13.8GB) to shared storage.  
Skipped automatically if `config.json` already exists in the model directory.

In [ ]:
@ray.remote(num_cpus=10)
def download_model():
    """Downloads the entire model repository to shared storage."""
    log(f"\n[Model] Checking storage path: {STORAGE_PATH}")
    log(f"[Model] Downloading {MODEL_ID} to {MODEL_SAVE_PATH}...")

    hf_token = os.environ.get("HF_TOKEN")
    masked_token = f"{hf_token[:4]}...{hf_token[-4:]}" if hf_token else "NOT_SET"
    log(f"[Model] Parameters: repo_id={MODEL_ID}, local_dir={MODEL_SAVE_PATH}, token={masked_token}")

    try:
        token_arg = os.environ.get("HF_TOKEN") or False
        snapshot_download(
            repo_id=MODEL_ID,
            local_dir=MODEL_SAVE_PATH,
            token=token_arg,
            local_dir_use_symlinks=False,
        )
        log(f"[Model] Success! Model saved to {MODEL_SAVE_PATH}")
    except Exception as e:
        log(f"[Model] Error downloading model: {e}")
        raise


# Check if model already exists
model_exists = os.path.exists(MODEL_SAVE_PATH) and "config.json" in os.listdir(MODEL_SAVE_PATH)

if model_exists:
    log(f"[Skip] Model already exists at {MODEL_SAVE_PATH} (config.json found)")
else:
    log(f"[Step 3] Downloading model...")
    ray.get(download_model.remote())

## Step 4: Download Dataset (Distributed)

Distributes the download of Fineweb-Edu parquet files across the cluster using Ray Data.  
Skipped automatically if 140+ parquet files already exist.

In [ ]:
def download_file_wrapper(batch):
    """Ray Data map function: Downloads a batch of files."""
    worker_id = f"{ray.get_runtime_context().get_task_id()}"
    node_ip = ray.util.get_node_ip_address()
    worker_tag = f"Worker {worker_id[-8:]}@{node_ip}"

    try:
        filenames = batch.get("item", [])
    except AttributeError:
        filenames = batch

    file_names = []
    statuses = []
    size_bytes_list = []

    token_arg = os.environ.get("HF_TOKEN") or False
    log(f"[{worker_tag}] Starting batch of {len(filenames)} files")

    for filename in filenames:
        try:
            log(f"[{worker_tag}] Processing: {filename}")
            file_path = os.path.join(DATASET_SAVE_PATH, filename)

            if os.path.exists(file_path):
                fsize = os.path.getsize(file_path)
                log(f"[{worker_tag}] Skipped (exists, {fsize/(1024*1024):.1f} MB): {filename}")
                file_names.append(filename)
                statuses.append("skipped")
                size_bytes_list.append(fsize)
                continue

            hf_hub_download(
                repo_id=DATASET_ID,
                filename=filename,
                repo_type="dataset",
                local_dir=DATASET_SAVE_PATH,
                token=token_arg
            )
            fsize = os.path.getsize(file_path) if os.path.exists(file_path) else 0
            log(f"[{worker_tag}] Downloaded ({fsize/(1024*1024):.1f} MB): {filename}")
            file_names.append(filename)
            statuses.append("downloaded")
            size_bytes_list.append(fsize)
        except Exception as e:
            log(f"[{worker_tag}] Failed: {filename} - {e}")
            file_names.append(filename)
            statuses.append("failed")
            size_bytes_list.append(0)

    batch_total_mb = sum(size_bytes_list) / (1024 * 1024)
    log(f"[{worker_tag}] Batch complete: {len(file_names)} files, {batch_total_mb:.1f} MB total")
    return {"file": file_names, "status": statuses, "size_bytes": size_bytes_list}


def download_dataset_distributed():
    """Orchestrates distributed download of dataset files using Ray Data."""
    log(f"\n[Dataset] Fetching file list for {DATASET_ID} ({DATASET_SUBSET})...")

    try:
        all_files = list_repo_files(
            repo_id=DATASET_ID,
            repo_type="dataset",
            token=os.environ.get("HF_TOKEN")
        )
    except Exception as e:
        log(f"[Dataset] Error listing repo files: {e}")
        return

    log(f"[Dataset] Total files in repo: {len(all_files)}")

    target_files = [f for f in all_files if f.endswith(".parquet") and DATASET_SUBSET in f]
    log(f"[Dataset] Filtered to {len(target_files)} parquet files matching '{DATASET_SUBSET}'.")
    log("[Dataset] Triggering distributed download via Ray Data...")

    start_time = time.time()
    input_data = [{"item": f} for f in target_files]
    ds = ray.data.from_items(input_data)

    num_files = len(target_files)
    batch_sz = max(1, num_files // 20)
    log(f"[Dataset] Distributing {num_files} files in batches of {batch_sz}")

    downloaded_ds = ds.map_batches(
        download_file_wrapper,
        batch_size=batch_sz,
        num_cpus=1,
        concurrency=20
    )

    all_results = downloaded_ds.take_all()
    duration = time.time() - start_time

    success_count = sum(1 for r in all_results if r["status"] != "failed")
    failed_count = sum(1 for r in all_results if r["status"] == "failed")
    total_bytes = sum(r.get("size_bytes", 0) for r in all_results)
    total_gb = total_bytes / (1024 ** 3)

    log(f"[Dataset] Summary: Processed {len(all_results)} files.")
    log(f"[Dataset] Total data volume: {total_gb:.2f} GB ({total_bytes:,} bytes)")
    log(f"[Dataset] Successfully downloaded {success_count}/{len(target_files)} files in {duration:.2f} seconds.")
    if duration > 0:
        log(f"[Dataset] Throughput: {total_gb/duration*1024:.1f} MB/s")
    if failed_count > 0:
        failed_files = [r["file"] for r in all_results if r["status"] == "failed"]
        log(f"[Dataset] WARNING: {failed_count} files failed: {failed_files}")


# Check if dataset already exists
dataset_exists = False
if os.path.exists(DATASET_SAVE_PATH):
    parquet_count = sum(1 for r, d, fs in os.walk(DATASET_SAVE_PATH) for f in fs if f.endswith(".parquet"))
    dataset_exists = parquet_count >= 140

if dataset_exists:
    log(f"[Skip] Dataset already exists at {DATASET_SAVE_PATH} ({parquet_count} parquet files)")
else:
    log(f"[Step 4] Downloading dataset (distributed)...")
    download_dataset_distributed()

## Step 5: Validate Downloads

In [ ]:
def validate_downloads():
    """Verifies that the model and dataset files are present in shared storage."""
    log("\n[Validation] Verifying downloads...")
    validation_failed = False

    # Validate Model
    if os.path.exists(MODEL_SAVE_PATH):
        model_files = os.listdir(MODEL_SAVE_PATH)
        if "config.json" in model_files:
            log(f"[Validation] Model verification PASSED. Found config.json and {len(model_files)-1} other files in {MODEL_SAVE_PATH}.")
        elif len(model_files) > 0:
            log(f"[Validation] Model verification WARNING. Directory not empty but 'config.json' missing. Files found: {len(model_files)}")
        else:
            log(f"[Validation] Model verification FAILED. Directory {MODEL_SAVE_PATH} is empty.")
            validation_failed = True
    else:
        log(f"[Validation] Model verification FAILED. Directory {MODEL_SAVE_PATH} does not exist.")
        validation_failed = True

    # Validate Dataset
    if os.path.exists(DATASET_SAVE_PATH):
        parquet_files = []
        for root, dirs, files in os.walk(DATASET_SAVE_PATH):
            for f in files:
                if f.endswith(".parquet"):
                    parquet_files.append(os.path.relpath(os.path.join(root, f), DATASET_SAVE_PATH))
        if len(parquet_files) > 0:
            log(f"[Validation] Dataset verification PASSED. Found {len(parquet_files)} parquet files under {DATASET_SAVE_PATH}.")
        else:
            log(f"[Validation] Dataset verification FAILED. No parquet files found under {DATASET_SAVE_PATH}.")
            validation_failed = True
    else:
        log(f"[Validation] Dataset verification FAILED. Directory {DATASET_SAVE_PATH} does not exist.")
        validation_failed = True

    if validation_failed:
        log("\n[Validation] One or more validations failed!")
        raise RuntimeError("Download validation failed")
    else:
        log("\n[Validation] All validations passed successfully.")

validate_downloads()

## Step 6: Distributed Training

Launches 15 workers, each loading the full Mistral-7B model and training for 10 epochs.  
Per-epoch checkpoints are saved with `save_pretrained(max_shard_size="4GB")`.

In [ ]:
@ray.remote(num_cpus=0)
class TrainingTracker:
    """Ray actor that collects status updates from all training workers."""
    def __init__(self, num_workers, num_epochs):
        self.num_workers = num_workers
        self.num_epochs = num_epochs
        self.status = {}
        for i in range(num_workers):
            self.status[i] = {
                "phase": "starting", "epoch": 0, "batches": 0,
                "bytes_read_mb": 0.0, "last_epoch_time": 0.0,
                "last_ckpt_time": 0.0, "total_time": 0.0,
                "node_ip": "", "done": False,
            }

    def update(self, worker_idx, phase, epoch=0, batches=0, bytes_read_mb=0.0,
              epoch_time=0.0, ckpt_time=0.0, total_time=0.0, node_ip="", done=False):
        self.status[worker_idx] = {
            "phase": phase, "epoch": epoch, "batches": batches,
            "bytes_read_mb": bytes_read_mb, "last_epoch_time": epoch_time,
            "last_ckpt_time": ckpt_time, "total_time": total_time,
            "node_ip": node_ip, "done": done,
        }

    def get_status(self):
        return dict(self.status)


@ray.remote(num_cpus=30)
def train_worker(worker_idx, assigned_files, tracker, num_epochs=10):
    """
    Distributed training worker. Loads the entire model using from_pretrained,
    simulates training, and checkpoints with save_pretrained.
    """
    from transformers import AutoModelForCausalLM, AutoTokenizer

    node_ip = ray.util.get_node_ip_address()
    tag = f"Train-Worker-{worker_idx}@{node_ip}"
    log(f"\n[{tag}] Starting training simulation...")
    log(f"[{tag}] Data: {len(assigned_files)} files, {num_epochs} epochs")

    tracker.update.remote(worker_idx, "loading_model", node_ip=node_ip)

    try:
        log(f"[{tag}] Loading model and tokenizer from {MODEL_SAVE_PATH}...")
        load_start = time.time()
        model = AutoModelForCausalLM.from_pretrained(MODEL_SAVE_PATH)
        tokenizer = AutoTokenizer.from_pretrained(MODEL_SAVE_PATH)
        load_time = time.time() - load_start

        num_params = sum(p.numel() for p in model.parameters())
        mem_mb = sum(p.nbytes for p in model.parameters()) / (1024 * 1024)
        model.train()
        log(f"[{tag}] Model loaded: {num_params:,} params, {mem_mb:.1f} MB in {load_time:.2f}s")
        tracker.update.remote(worker_idx, "training", epoch=0, node_ip=node_ip)

    except Exception as e:
        log(f"[{tag}] Failed to load model: {e}")
        import traceback
        traceback.print_exc()
        tracker.update.remote(worker_idx, "FAILED", node_ip=node_ip)
        return False

    # Training loop
    worker_checkpoint_dir = os.path.join(CHECKPOINT_SAVE_PATH, f"worker_{worker_idx}")
    os.makedirs(worker_checkpoint_dir, exist_ok=True)

    total_batches_processed = 0
    total_bytes_read = 0
    epoch_times = []
    chunk_size = 1024 * 1024

    for epoch in range(1, num_epochs + 1):
        epoch_start = time.time()
        batches_this_epoch = 0
        bytes_this_epoch = 0

        for data_file in assigned_files:
            full_path = os.path.join(DATASET_SAVE_PATH, data_file)
            if not os.path.exists(full_path):
                log(f"[{tag}] Warning: Data file missing: {data_file}")
                continue

            file_bytes = 0
            try:
                with open(full_path, 'rb') as f:
                    while True:
                        chunk = f.read(chunk_size)
                        if not chunk:
                            break
                        file_bytes += len(chunk)
            except Exception as e:
                log(f"[{tag}] Error reading {data_file}: {e}")
                continue

            file_size_mb = file_bytes / (1024 * 1024)
            num_batches = max(1, int(file_size_mb / 20))

            for _ in range(num_batches):
                time.sleep(0.01)
                batches_this_epoch += 1
                total_batches_processed += 1

            bytes_this_epoch += file_bytes
            total_bytes_read += file_bytes

        epoch_time = time.time() - epoch_start
        epoch_times.append(epoch_time)

        # Checkpoint at end of each epoch
        ckpt_start = time.time()
        epoch_ckpt_dir = os.path.join(worker_checkpoint_dir, f"epoch_{epoch}")
        model.save_pretrained(epoch_ckpt_dir, max_shard_size="4GB")
        tokenizer.save_pretrained(epoch_ckpt_dir)

        epoch_meta = {
            "worker_idx": worker_idx, "epoch": epoch,
            "num_params": num_params, "batches_processed": batches_this_epoch,
            "total_batches": total_batches_processed,
            "epoch_time_s": round(epoch_time, 2),
            "files_processed": len(assigned_files),
        }
        with open(os.path.join(epoch_ckpt_dir, f"epoch_{epoch}_meta.json"), "w") as f:
            json.dump(epoch_meta, f, indent=2)

        ckpt_time = time.time() - ckpt_start
        log(f"[{tag}] Epoch {epoch}/{num_epochs}: {batches_this_epoch} batches, "
              f"{bytes_this_epoch / (1024*1024):.1f} MB read, "
              f"{epoch_time:.2f}s training, {ckpt_time:.2f}s checkpoint")

        tracker.update.remote(
            worker_idx, "training", epoch=epoch, batches=total_batches_processed,
            bytes_read_mb=round(total_bytes_read / (1024*1024), 1),
            epoch_time=round(epoch_time, 2), ckpt_time=round(ckpt_time, 2),
            total_time=round(sum(epoch_times), 2), node_ip=node_ip
        )

    total_time = sum(epoch_times)
    log(f"[{tag}] Training complete: {num_epochs} epochs, {total_batches_processed} total batches, "
          f"{total_bytes_read / (1024*1024*1024):.2f} GB read, {total_time:.2f}s total")
    tracker.update.remote(
        worker_idx, "done", epoch=num_epochs, batches=total_batches_processed,
        bytes_read_mb=round(total_bytes_read / (1024*1024), 1),
        total_time=round(total_time, 2), node_ip=node_ip, done=True
    )
    return True

In [ ]:
def distribute_training(num_epochs=10):
    """Orchestrates distributed training across 15 workers."""
    num_workers = 15
    log(f"\n[Training] Starting distributed training: {num_workers} workers, {num_epochs} epochs")

    # Discover and distribute training data
    parquet_files = []
    for root, dirs, files in os.walk(DATASET_SAVE_PATH):
        for f in files:
            if f.endswith(".parquet"):
                parquet_files.append(os.path.relpath(os.path.join(root, f), DATASET_SAVE_PATH))
    parquet_files.sort()
    log(f"[Training] Found {len(parquet_files)} training data files")

    if not parquet_files:
        raise RuntimeError("No training data files found!")

    worker_files = [[] for _ in range(num_workers)]
    for i, f in enumerate(parquet_files):
        worker_files[i % num_workers].append(f)

    for i in range(num_workers):
        log(f"[Training] Worker {i}: {len(worker_files[i])} data files")

    # Create status tracker and launch training tasks
    tracker = TrainingTracker.remote(num_workers, num_epochs)
    start_time = time.time()
    futures = [
        train_worker.remote(i, worker_files[i], tracker, num_epochs)
        for i in range(num_workers)
    ]

    # Poll tracker every 5 seconds and print status table
    done_refs = set()
    while len(done_refs) < len(futures):
        ready, pending = ray.wait(
            [f for f in futures if f not in done_refs],
            timeout=5.0,
            num_returns=len(futures) - len(done_refs)
        )
        done_refs.update(ready)

        status = ray.get(tracker.get_status.remote())
        elapsed = time.time() - start_time
        log(f"\n{'='*90}")
        log(f"  TRAINING STATUS  (elapsed: {elapsed:.0f}s, {len(done_refs)}/{num_workers} workers done)")
        log(f"{'='*90}")
        log(f"  {'Worker':<10} {'Node IP':<18} {'Phase':<15} {'Epoch':<10} {'Batches':<10} {'Data (GB)':<10} {'Time(s)':<10}")
        log(f"  {'-'*10} {'-'*18} {'-'*15} {'-'*10} {'-'*10} {'-'*10} {'-'*10}")
        for w in range(num_workers):
            s = status[w]
            epoch_str = f"{s['epoch']}/{num_epochs}" if s['epoch'] > 0 else "-"
            data_gb = f"{s['bytes_read_mb']/1024:.1f}" if s['bytes_read_mb'] > 0 else "-"
            batches = str(s['batches']) if s['batches'] > 0 else "-"
            total_t = f"{s['total_time']:.0f}" if s['total_time'] > 0 else "-"
            phase = "DONE \u2713" if s['done'] else s['phase']
            log(f"  W-{w:<7} {s['node_ip']:<18} {phase:<15} {epoch_str:<10} {batches:<10} {data_gb:<10} {total_t:<10}")
        log(f"{'='*90}")

    results = ray.get(futures)
    total_time = time.time() - start_time

    success_count = sum(1 for r in results if r)
    log(f"\n[Training] Complete: {success_count}/{num_workers} workers succeeded in {total_time:.2f}s")

    if success_count != num_workers:
        raise RuntimeError(f"{num_workers - success_count} workers failed!")


log(f"[Step 6] Distributed Training (full model per worker, 10 epochs)")
distribute_training(num_epochs=10)

## Step 7: Model Consolidation

Aggregates final-epoch checkpoints from all 15 workers.  
Runs on a worker node (192GB RAM) to avoid head node OOM.  
Saves final model with `save_pretrained(max_shard_size="4GB")`.

In [ ]:
@ray.remote(num_cpus=0)
class ConsolidationTracker:
    """Ray actor that tracks consolidation progress."""
    def __init__(self, num_workers):
        self.num_workers = num_workers
        self.phase = "discovering"
        self.workers_found = 0
        self.workers_loaded = 0
        self.load_times = {}
        self.current_worker = -1
        self.avg_time = 0.0
        self.save_time = 0.0
        self.total_time = 0.0
        self.node_ip = ""
        self.error = None

    def update_phase(self, phase, **kwargs):
        self.phase = phase
        for k, v in kwargs.items():
            setattr(self, k, v)

    def worker_loaded(self, worker_idx, load_time):
        self.load_times[worker_idx] = load_time
        self.workers_loaded = len(self.load_times)
        self.current_worker = worker_idx

    def get_status(self):
        return {
            "phase": self.phase, "num_workers": self.num_workers,
            "workers_found": self.workers_found,
            "workers_loaded": self.workers_loaded,
            "load_times": dict(self.load_times),
            "current_worker": self.current_worker,
            "avg_time": self.avg_time, "save_time": self.save_time,
            "total_time": self.total_time, "node_ip": self.node_ip,
            "error": self.error,
        }


@ray.remote(num_cpus=30)
def consolidate_model(tracker, num_epochs=10):
    """
    Aggregates final-epoch checkpoints from all workers.
    Uses last worker's checkpoint as the final model.
    """
    import shutil
    import gc
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch

    num_workers = 15
    final_model_dir = os.path.join(CHECKPOINT_SAVE_PATH, "final_model")
    node_ip = ray.util.get_node_ip_address()

    log(f"\n{'='*80}")
    log(f"[Aggregation] Starting checkpoint aggregation from {num_workers} workers (epoch {num_epochs})")
    log(f"[Aggregation] Running on worker node {node_ip} to avoid head node OOM (8GB limit)")
    log(f"{'='*80}")

    tracker.update_phase.remote("discovering", node_ip=node_ip)

    # Discover available checkpoints
    loaded_workers = []
    for worker_idx in range(num_workers):
        epoch_ckpt_dir = os.path.join(
            CHECKPOINT_SAVE_PATH, f"worker_{worker_idx}", f"epoch_{num_epochs}"
        )
        if os.path.exists(epoch_ckpt_dir):
            loaded_workers.append(worker_idx)
            log(f"  Worker {worker_idx}: checkpoint found")
        else:
            log(f"  Worker {worker_idx}: checkpoint MISSING")

    if not loaded_workers:
        log(f"[Aggregation] ERROR: No checkpoints found!")
        tracker.update_phase.remote("error", error="No checkpoints found")
        return None

    log(f"\n[Aggregation] Found {len(loaded_workers)}/{num_workers} worker checkpoints")
    log(f"[Aggregation] Simulating consolidation: iterate all checkpoints, load only last worker's")
    tracker.update_phase.remote("loading", workers_found=len(loaded_workers))

    # Simulate loading for all workers, actually load only the last one
    aggregation_start = time.time()
    load_times = []
    last_worker_idx = loaded_workers[-1]
    last_ckpt_dir = os.path.join(
        CHECKPOINT_SAVE_PATH, f"worker_{last_worker_idx}", f"epoch_{num_epochs}"
    )

    for i, worker_idx in enumerate(loaded_workers):
        epoch_ckpt_dir = os.path.join(
            CHECKPOINT_SAVE_PATH, f"worker_{worker_idx}", f"epoch_{num_epochs}"
        )
        load_start = time.time()

        if worker_idx == last_worker_idx:
            log(f"  Loading worker {worker_idx} checkpoint (actual load)...")
            final_model = AutoModelForCausalLM.from_pretrained(epoch_ckpt_dir)
            lt = time.time() - load_start
            log(f"  Loaded worker {worker_idx} checkpoint ({lt:.2f}s) [{i+1}/{len(loaded_workers)}] (REAL)")
        else:
            ckpt_files = os.listdir(epoch_ckpt_dir)
            ckpt_size_mb = sum(
                os.path.getsize(os.path.join(epoch_ckpt_dir, f))
                for f in ckpt_files if os.path.isfile(os.path.join(epoch_ckpt_dir, f))
            ) / (1024 * 1024)
            time.sleep(0.5)
            lt = time.time() - load_start
            log(f"  Simulated worker {worker_idx} checkpoint ({ckpt_size_mb:.1f} MB, {len(ckpt_files)} files) [{i+1}/{len(loaded_workers)}]")

        load_times.append(lt)
        tracker.worker_loaded.remote(worker_idx, round(lt, 2))

    log(f"\n[Aggregation] Using last worker's (worker {last_worker_idx}) checkpoint as final model")
    tracker.update_phase.remote("averaging")
    avg_start = time.time()
    avg_time = time.time() - avg_start
    log(f"[Aggregation] Consolidation simulation complete ({avg_time:.2f}s)")

    # Save final model (sharded ~4GB safetensors)
    log(f"[Aggregation] Saving final model with max_shard_size=4GB...")
    tracker.update_phase.remote("saving", avg_time=round(avg_time, 2))
    save_start = time.time()

    if os.path.exists(final_model_dir):
        shutil.rmtree(final_model_dir)
    os.makedirs(final_model_dir, exist_ok=True)

    final_model.save_pretrained(final_model_dir, max_shard_size="4GB")
    tokenizer = AutoTokenizer.from_pretrained(last_ckpt_dir)
    tokenizer.save_pretrained(final_model_dir)

    save_time = time.time() - save_start
    log(f"[Aggregation] Model saved in {save_time:.2f}s")

    # Compute stats
    num_params = sum(p.numel() for p in final_model.parameters())
    del final_model
    gc.collect()

    copied_files = os.listdir(final_model_dir)
    total_size_mb = sum(
        os.path.getsize(os.path.join(final_model_dir, f))
        for f in copied_files if os.path.isfile(os.path.join(final_model_dir, f))
    ) / (1024 * 1024)
    num_shards = len([f for f in copied_files if f.endswith(".safetensors")])
    log(f"[Aggregation] Saved {num_shards} shard(s), total {total_size_mb:.1f} MB")

    consolidation_meta = {
        "num_workers_aggregated": len(loaded_workers),
        "final_epoch": num_epochs,
        "num_params": num_params,
        "total_size_mb": round(total_size_mb, 1),
        "num_shards": num_shards,
        "avg_load_time_s": round(sum(load_times) / len(load_times), 2),
        "averaging_time_s": round(avg_time, 2),
        "save_time_s": round(save_time, 2),
        "total_time_s": round(time.time() - aggregation_start, 2),
        "source_model": MODEL_ID,
        "files": copied_files,
    }
    with open(os.path.join(final_model_dir, "consolidation_meta.json"), "w") as f:
        json.dump(consolidation_meta, f, indent=2)

    total_time = time.time() - aggregation_start
    log(f"\n[Aggregation] Final model saved to {final_model_dir}")
    log(f"[Aggregation] {num_params:,} params, {total_size_mb:.1f} MB, aggregated in {total_time:.2f}s")
    log(f"{'='*80}")

    tracker.update_phase.remote(
        "done", save_time=round(save_time, 2), total_time=round(total_time, 2)
    )
    return consolidation_meta

In [ ]:
log(f"[Step 7] Final Model Consolidation (on worker node)")
consolidation_tracker = ConsolidationTracker.remote(15)
consolidation_future = consolidate_model.remote(consolidation_tracker, num_epochs=10)

# Poll consolidation tracker every 5 seconds
while True:
    ready, _ = ray.wait([consolidation_future], timeout=5.0)
    if ready:
        break

    cs = ray.get(consolidation_tracker.get_status.remote())
    log(f"\n{'='*80}")
    log(f"  CONSOLIDATION STATUS  (phase: {cs['phase']})")
    log(f"{'='*80}")
    log(f"  Node: {cs['node_ip']}")
    log(f"  Checkpoints found: {cs['workers_found']}/{cs['num_workers']}")
    log(f"  Checkpoints loaded: {cs['workers_loaded']}/{cs['workers_found']}")
    if cs['load_times']:
        log(f"  {'Worker':<10} {'Load Time':<12} {'Status':<10}")
        log(f"  {'-'*10} {'-'*12} {'-'*10}")
        for w in range(cs['num_workers']):
            if w in cs['load_times']:
                log(f"  W-{w:<7} {cs['load_times'][w]:.2f}s       loaded")
            elif w == cs['current_worker'] + 1 and cs['phase'] == 'loading':
                log(f"  W-{w:<7} ...          loading")
            elif cs['phase'] == 'loading' and w > cs.get('workers_loaded', 0):
                log(f"  W-{w:<7} -            pending")
    if cs['phase'] == 'averaging':
        log(f"  Averaging weights across {cs['workers_loaded']} checkpoints...")
    elif cs['phase'] == 'saving':
        log(f"  Avg time: {cs['avg_time']:.2f}s | Saving final model...")
    log(f"{'='*80}")

aggregation_stats = ray.get(consolidation_future)
log("[Step 7] Consolidation complete!")

## Step 8: Job Summary & Cleanup

In [ ]:
@ray.remote(num_cpus=1)
def _collect_node_logs():
    """Collects local log files from a worker node."""
    import glob
    hostname = socket.gethostname()
    log_files = glob.glob(os.path.join(LOG_DIR, "*.log"))
    logs = {}
    for lf in log_files:
        try:
            with open(lf, "r") as f:
                logs[os.path.basename(lf)] = f.read()
        except Exception:
            pass
    return {"hostname": hostname, "logs": logs}


def collect_worker_logs():
    """Gathers log files from all worker nodes and saves to shared storage."""
    log("\n[Logs] Collecting worker logs from all nodes...")
    logs_output_dir = os.path.join(STORAGE_PATH, "job_logs", str(_JOB_TIMESTAMP))
    os.makedirs(logs_output_dir, exist_ok=True)

    try:
        import shutil
        if os.path.exists(LOG_FILE):
            shutil.copy2(LOG_FILE, os.path.join(logs_output_dir, os.path.basename(LOG_FILE)))
            log(f"[Logs] Saved head node log: {os.path.basename(LOG_FILE)}")
    except Exception as e:
        log(f"[Logs] Warning: Failed to save head node log: {e}")

    try:
        num_tasks = 20
        futures = [_collect_node_logs.remote() for _ in range(num_tasks)]
        results = ray.get(futures, timeout=60)

        seen_hosts = set()
        for result in results:
            hostname = result["hostname"]
            if hostname in seen_hosts:
                continue
            seen_hosts.add(hostname)
            for filename, content in result["logs"].items():
                out_path = os.path.join(logs_output_dir, filename)
                if not os.path.exists(out_path):
                    with open(out_path, "w") as f:
                        f.write(content)

        log(f"[Logs] Collected logs from {len(seen_hosts)} nodes to {logs_output_dir}")
    except Exception as e:
        log(f"[Logs] Warning: Failed to collect some worker logs: {e}")


# Print job summary
log(f"\n{'='*80}")
log(f"{'TRAINING JOB SUMMARY':^80}")
log(f"{'='*80}")
log(f"")
log(f"{'Model:':<40} {MODEL_ID}")
log(f"{'Dataset:':<40} {DATASET_ID} ({DATASET_SUBSET})")
log(f"{'Workers:':<40} 15")
log(f"{'Epochs:':<40} 10")
log(f"")
if aggregation_stats:
    log(f"{'Aggregation:':<40}")
    log(f"  - Workers aggregated:                {aggregation_stats['num_workers_aggregated']}")
    log(f"  - Avg checkpoint load time:          {aggregation_stats['avg_load_time_s']:.2f}s")
    log(f"  - Weight averaging time:             {aggregation_stats['averaging_time_s']:.2f}s")
    log(f"  - Final model save time:             {aggregation_stats['save_time_s']:.2f}s")
    log(f"  - Final model size:                  {aggregation_stats['total_size_mb']:.1f} MB")
log(f"")
log(f"{'='*80}")
log(f"[Success] Job completed successfully.")

# Collect worker logs
collect_worker_logs()